In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
(train_images, train_labels), (test_images, _) = tf.keras.datasets.mnist.load_data()

In [ ]:
train_labels = [[label] for label in train_labels]

In [ ]:
train_labels = tf.keras.preprocessing.sequence.pad_sequences(train_labels, maxlen=3, padding='post', truncating='post')

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))

In [ ]:
buffer = tf.data.AUTOTUNE
dataset = dataset.batch(32).prefetch(buffer_size=buffer)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Shape: height, width, channels.
input_shape = (28, 28, 1)
input_image = layers.Input(shape=input_shape)

x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_image)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)

# Reshape CNN features for the RNN.
x = layers.Reshape(target_shape=(-1, x.shape[2] * x.shape[3]))(x)

x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)

x = layers.Dense(11)(x)  # 10 digits + CTC blank.

model = models.Model(input_image, x)

In [ ]:
model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001), loss='CTC', metrics=['accuracy'])

In [ ]:
model.fit(dataset, epochs=20)

In [ ]:
inference_model = tf.keras.Model(inputs=input_image, outputs=x)

def greedy_decoder(logits, input_lengths):
    probabilities = tf.nn.softmax(logits)
    decoded, _ = tf.nn.ctc_greedy_decoder(
        inputs=tf.transpose(probabilities, perm=[1, 0, 2]),  # Time-major format for CTC.
        sequence_length=input_lengths
    )
    return tf.sparse.to_dense(decoded[0])

test_image = test_images[0:1]
test_input_length = tf.constant([test_image.shape[0]])  # Sequence length for CTC.

test_logits = inference_model.predict(test_image)

plt.imshow(test_image[0], cmap='gray')
plt.title("Input Test Image")
plt.axis('off')
plt.show()

decoded_sequence = greedy_decoder(test_logits, test_input_length)
print("Decoded sequence:", decoded_sequence.numpy())